In [6]:
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.cluster import DBSCAN
from difflib import SequenceMatcher
from Levenshtein import distance as levenshtein_distance
from itertools import combinations
from collections import defaultdict
import pickle
import os
from Soccer_OCEL.role_objects import assign_roles_multi
import Soccer_OCEL.utils as utils
import pandas as pd

In [2]:
def encode_sequences(sequences: list[list[str]]) -> tuple[list[tuple], LabelEncoder]:
    le = LabelEncoder()
    all_labels = [label for seq in sequences for label in seq]
    le.fit(all_labels)
    encoded = [tuple(le.transform(seq)) for seq in sequences]
    return encoded, le


def build_distance_matrix_levenshtein(encoded_sequences: list[tuple]) -> np.ndarray:
    n = len(encoded_sequences)
    dist_matrix = np.zeros((n, n), dtype=np.float32)
    # convert to strings of tokens for levenshtein
    joined = [' '.join(map(str, seq)) for seq in encoded_sequences]
    for i, j in combinations(range(n), 2):
        d = levenshtein_distance(joined[i], joined[j])
        dist_matrix[i][j] = d
        dist_matrix[j][i] = d
    return dist_matrix


def build_distance_matrix_sequencematcher(encoded_sequences: list[tuple]) -> np.ndarray:
    n = len(encoded_sequences)
    dist_matrix = np.zeros((n, n), dtype=np.float32)
    for i, j in combinations(range(n), 2):
        sim = SequenceMatcher(None, encoded_sequences[i], encoded_sequences[j]).ratio()
        d = 1 - sim
        dist_matrix[i][j] = d
        dist_matrix[j][i] = d
    return dist_matrix


def cluster(dist_matrix: np.ndarray, eps: float, min_samples: int = 2) -> np.ndarray:
    labels = DBSCAN(eps=eps, min_samples=min_samples, metric='precomputed').fit(dist_matrix).labels_
    return labels


def inspect_clusters(labels: np.ndarray, sequences: list[list[str]]) -> dict:
    clusters = defaultdict(list)
    for idx, label in enumerate(labels):
        clusters[label].append(sequences[idx])
    return dict(clusters)


def print_clusters(clusters: dict, max_seqs: int = 3):
    for label, seqs in sorted(clusters.items()):
        tag = 'NOISE' if label == -1 else f'Cluster {label}'
        print(f"\n{tag} ({len(seqs)} sequences):")
        for s in seqs[:max_seqs]:
            print(f"  {s}")
        if len(seqs) > max_seqs:
            print(f"  ... and {len(seqs) - max_seqs} more")

In [3]:
#load all
path='data/28196177'
FD='DFL-CLU-00000P'
#games=get_games(path)
games=['J03WQQ', 'J03WR9', 'J03WPY', 'J03WOH', 'J03WOY']#Fortuna Düsseldorf
#games=['J03WQQ', 'J03WOH', 'J03WOY']#Fortuna Düsseldorf, won games
#games=['J03WR9', 'J03WPY']#Fortuna Düsseldorf lost game
GDs=[]
all_games=[]
for Game in games:
    with open(os.path.join('output','output_pkl',f"GameData_{Game}.pkl"), "rb") as f:
        GD_loaded = pickle.load(f)
    #GD_loaded.join_events(log='MOVEMENT')
    GD_loaded=assign_roles_multi(GD_loaded, team=True)
    #GD_loaded.encode_pass_distance()
    GD_loaded.format_log(log='ALL')
    GDs.append(GD_loaded)

In [21]:
allgame_df

,Player,attribute:session,concept:name,attribute:frame,end_frame,movement,dx,dy,angle,angle_attacking,...,left_a,right,left,type,playmaker_h,playmaker_a,playmaker,tID,n,movement_type
0,DFL-OBJ-00003X,1,Forward-Right_long,0,14,2.793078,-2.35,-1.50,212.855129,36.869898,...,None,None,None,Unknown,None,None,None,DFL-CLU-00000P,14,long
1,DFL-OBJ-0000F8,1,Forward_long,0,11,1.106454,-1.06,-0.31,196.156842,12.528808,...,None,DFL-OBJ-0000F8,None,Unknown,None,None,None,DFL-CLU-00000P,11,long
2,DFL-OBJ-002FXT,1,Forward-Left_long,0,16,3.412320,-2.94,1.71,149.391273,322.431408,...,None,None,None,playmaker,DFL-OBJ-002FXT,None,DFL-OBJ-002FXT,DFL-CLU-00000P,16,long
3,DFL-OBJ-J0130T,1,Forward-Right_long,0,42,6.563017,-5.71,-3.21,209.637598,33.690068,...,None,None,DFL-OBJ-J0130T,Unknown,None,None,None,DFL-CLU-00000P,42,long
4,DFL-OBJ-002GM1,1,Forward_long,3,29,2.752844,-2.71,-0.43,188.649169,0.000000,...,None,None,DFL-OBJ-002GM1,Unknown,None,None,None,DFL-CLU-00000P,26,long
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
214969,DFL-OBJ-0028FW,2,Backward-Left_short,78445,78445,0.152315,-0.14,-0.06,203.198591,203.198591,...,None,None,None,Unknown,None,None,None,DFL-CLU-00000P,0,short
214970,DFL-OBJ-0028FW,2,Backward_short,78446,78446,0.161555,-0.15,-0.06,201.801409,201.801409,...,None,None,None,Unknown,None,None,None,DFL-CLU-00000P,0,short
214971,DFL-OBJ-0028FW,2,Backward-Left_short,78447,78447,0.152315,-0.14,-0.06,203.198591,203.198591,...,None,None,None,Unknown,None,None,None,DFL-CLU-00000P,0,short
214972,DFL-OBJ-0028FW,2,Backward_short,78448,78448,0.148661,-0.14,-0.05,199.653824,199.653824,...,None,None,None,Unknown,None,None,None,DFL-CLU-00000P,0,short


In [ ]:
all_games=[]
for GD_loaded in GDs:
    #GD_loaded.encode_pass_distance()
    df=GD_loaded.events.query('`case:concept:name`.notna()').copy()
    all_games.append(df)

In [5]:
all_games=[]
for GD_loaded in GDs:
    df=GD_loaded.movement_events.query('`case:concept:name`.notna()').copy()
    df['tID'] = df['Player'].apply(lambda x: utils.get_tID_from_pID(x, GD_loaded.team_sheets_df))
    all_games.append(df)

In [ ]:
all_games=[]
for GD_loaded in GDs:
    df=GD_loaded.positional_events.query('`case:concept:name`.notna()').copy()
    df['tID'] = df['Player'].apply(lambda x: utils.get_tID_from_pID(x, GD_loaded.team_sheets_df))
    all_games.append(df)

In [7]:
FD_log=[]
for i, game_log in enumerate(all_games):
    #teamside=GDs[i].team_sheets_df.query('tID==@FD')['Home_Away'].values[0]
    #FD_log.append(game_log[game_log['case:concept:name'].str.contains(teamside, na=False)])
    temp=game_log.query('tID==@FD').copy()
    temp['case:concept:name']=temp['case:concept:name'] + '_' + temp['Player']
    FD_log.append(temp)
    #FD_log.append(game_log[(game_log['tID'] == FD) & (game_log['case:concept:name'].str.contains(teamside, na=False))])
allgame_df=pd.concat(FD_log).sort_values(["attribute:game","attribute:session", "attribute:frame"]).reset_index(drop=True)

In [15]:
allgame_df['n']=allgame_df['end_frame']-allgame_df['attribute:frame']
short_thr = allgame_df['n'].quantile(0.33)
long_thr  = allgame_df['n'].quantile(0.66)

def classify(d):
    if d <= short_thr:
        return "short"
    elif d <= long_thr:
        return "medium"
    else:
        return "long"

allgame_df['movement_type'] = allgame_df['n'].apply(classify)

allgame_df['concept:name'] = allgame_df['concept:name'].str.cat(allgame_df['movement_type'], sep='_')

In [16]:
grouped = allgame_df.groupby('case:concept:name')['concept:name'].apply(list)
case_ids = grouped.index.tolist()
sequences = grouped.tolist()
encoded_sequences, le = encode_sequences(sequences)

In [17]:
# levenshtein
dist_lev = build_distance_matrix_levenshtein(encoded_sequences)
labels_lev = cluster(dist_lev, eps=3, min_samples=2)
clusters_lev = inspect_clusters(labels_lev, sequences)
print("=== Levenshtein Clustering ===")
print_clusters(clusters_lev)

=== Levenshtein Clustering ===

NOISE (7222 sequences):
  ['Backward-Left_medium', 'Backward_long', 'Backward-Right_short', 'Backward_short', 'Backward-Right_short', 'Backward_short', 'Backward-Right_medium', 'Backward_short', 'Backward-Right_short', 'Backward_medium', 'Backward-Right_short', 'Backward_short', 'Backward-Right_short', 'Backward_medium', 'Backward-Right_short', 'Backward_short', 'Backward-Right_short', 'Backward_medium', 'Backward-Right_short', 'Backward_long', 'Backward-Right_short']
  ['Forward_long', 'Forward-Right_short', 'Forward_long', 'Forward-Right_short', 'Forward_short', 'Forward-Right_short', 'Forward_short', 'Forward-Right_short', 'Forward_long', 'Forward-Right_short', 'Forward_short', 'Forward-Right_short', 'Forward_short', 'Forward-Right_short', 'Forward_short', 'Forward-Right_medium', 'Forward_short', 'Forward-Right_long', 'Forward_short', 'Forward-Right_short', 'Forward_short', 'Forward-Right_long']
  ['Backward-Right_short', 'Backward_short', 'Backward-R

In [18]:
# sequencematcher
dist_sm = build_distance_matrix_sequencematcher(encoded_sequences)
labels_sm = cluster(dist_sm, eps=0.3, min_samples=2)
clusters_sm = inspect_clusters(labels_sm, sequences)
print("=== SequenceMatcher Clustering ===")
print_clusters(clusters_sm)

=== SequenceMatcher Clustering ===

NOISE (4084 sequences):
  ['Forward-Left_long', 'Forward_short', 'Forward-Left_long', 'Forward_short', 'Forward-Left_short', 'Forward_medium', 'Forward-Left_short', 'Forward_medium', 'Forward-Left_short', 'Forward_short', 'Forward-Left_short', 'Forward_short', 'Forward-Left_medium', 'Forward_short', 'Forward-Left_medium', 'Forward_short', 'Forward-Left_short', 'Forward_short', 'Forward-Left_short', 'Forward_short', 'Forward-Left_short', 'Forward_short', 'Forward-Left_short', 'Forward_medium', 'Forward-Left_short', 'Forward_medium', 'Forward-Left_short', 'Forward_long', 'Forward-Left_short', 'Forward_long', 'Forward-Left_short', 'Forward_long', 'Forward-Left_medium', 'Forward_medium', 'Forward-Left_medium', 'Forward_medium', 'Forward-Left_short', 'Forward_short', 'Forward-Left_short', 'Forward_short', 'Forward-Left_short', 'Forward_medium', 'Forward-Left_medium', 'Forward_short', 'Forward-Left_short', 'Forward_short', 'Forward-Left_medium', 'Forward_s